# EMRI WDM TDI Benchmark

Benchmark notebook for dense WDM coefficient EMRI response generation and Gen-2 TDI assembly, focused on end-to-end runtime and accuracy.

In [12]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

import jax
jax.config.update('jax_enable_x64', True)

ROOT = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [ROOT, *ROOT.parents] if (p / 'WDM_LISAresponse').exists() and (p / 'WDM_GW_wavelets').exists()),
    ROOT,
 )
WDM_LISA_ROOT = PROJECT_ROOT / 'WDM_LISAresponse'
WDM_GW_ROOT = PROJECT_ROOT / 'WDM_GW_wavelets'
WDM_LISA_SRC = WDM_LISA_ROOT / 'src'
WDM_SRC = WDM_GW_ROOT / 'src'

for p in [str(WDM_LISA_SRC), str(WDM_SRC), str(PROJECT_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

import WDM
from lisaorbits.file import read_orbit_file
from lisaorbits import InterpolatedOrbits
from few.waveform import GenerateEMRIWaveform
from few.utils.constants import YRSID_SI

from WDM_LISAresponse.code.lisa_response import response as response_mod
from WDM_LISAresponse.code.lisa_response.waveforms import generate_windowed_padded_waveform
from WDM_LISAresponse.code.lisa_response.TDI import (
    gen2_sagnac_wdm,
    gen2_michelson_wdm,
    gen2_michelson_td,
 )
from WDM_LISAresponse.code.utils.utils import compute_wdm_cut_info, apply_wdm_cut, wdm_mismatch

In [13]:
BASELINE = {
    'dt': 10.0,
    'Nf': 2000,
    'L_trunc': 25,
    'q': 4,
    'calc_m0': True,
    'd': 4,
    'tl_tp_mode': 'interp',
    'tl_tp_interp_points': 2,
    'tl_tp_interp_kind': 'linear',
    'tl_tp_interp_pad': 0.0,
    'assembly_vmap': False,
    'batch_chunk': 32,
    'alpha': 0.05,
    'tukey_factor': 1.0,
}

T_OBS_YEARS = 0.25
N_SAMPLES = int(np.floor(T_OBS_YEARS * YRSID_SI / BASELINE['dt']))

EMRI_PARAMS = {
    'm1': 1e6, 'm2': 10.0, 'a': 0.9, 'p0': 8.5, 'e0': 0.3, 'x0': 1.0,
    'dist': 1.0, 'theta': np.pi / 2.0, 'phi': 0.0,
    'qK': np.pi / 3.0, 'phiK': np.pi / 3.0,
    'Phi_phi0': np.pi / 3.0, 'Phi_theta0': 0.0, 'Phi_r0': np.pi / 3.0,
    'eps': 1e-2,
}

In [14]:
td_gen = GenerateEMRIWaveform(
    'FastKerrEccentricEquatorialFlux',
    sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=True,
)

orbit_data = read_orbit_file(str(WDM_LISA_ROOT / 'examples' / 'esa-trailing-orbits.h5'))
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
orbits = InterpolatedOrbits(orbit_times, orbit_data.positions, orbit_data.velocities)
t0 = float(orbit_data.t0 + 1.0 * YRSID_SI)

In [15]:
def build_shift_kwargs(cfg):
    mode = str(cfg.get('tl_tp_mode', 'interp')).lower()
    if mode == 'exact':
        raise ValueError("tl_tp_mode='exact' is disabled in this notebook for memory-safe runs.")
    return {
        'use_jax': True,
        'tl_tp_mode': 'interp',
        'tl_tp_interp_points': int(cfg['tl_tp_interp_points']),
        'tl_tp_interp_kind': cfg['tl_tp_interp_kind'],
        'tl_tp_interp_pad': float(cfg['tl_tp_interp_pad']),
        'batch_chunk': int(cfg['batch_chunk']),
        'assembly_vmap': bool(cfg.get('assembly_vmap', False)),
    }

def generate_emri(cfg, n_samples):
    model_kwargs = dict(EMRI_PARAMS)
    model_kwargs['td_gen'] = td_gen
    return generate_windowed_padded_waveform(
        waveform_model='emri',
        t0=t0,
        dt=float(cfg['dt']),
        N=int(n_samples),
        model_kwargs=model_kwargs,
        window_function='tukey',
        window_kwargs={'alpha': cfg['alpha'], 'tukey_factor': cfg['tukey_factor']},
        wdm_Nf=int(cfg['Nf']),
        return_padding_info=True,
    )

def prebuild_waveform_payloads(cfg, sizes):
    payloads = {}
    for n in sizes:
        n_i = int(n)
        if n_i <= 0:
            continue
        payloads[n_i] = generate_emri(cfg, n_i)
    return payloads

In [16]:
def timed_ms(fn):
    t0_local = time.perf_counter()
    out = fn()
    return out, (time.perf_counter() - t0_local) * 1000.0

def run_pipeline(cfg, n_samples, waveform_payload=None):
    stages = {}
    if waveform_payload is None:
        (times_pad, hp, hc, pad_info), stages['waveform_ms'] = timed_ms(lambda: generate_emri(cfg, n_samples))
    else:
        times_pad, hp, hc, pad_info = waveform_payload
        stages['waveform_ms'] = 0.0

    shift_kwargs = build_shift_kwargs(cfg)
    one_arm_cache = {}
    shifts_cache = {}

    (wdm_obj, _), stages['one_arm_ms'] = timed_ms(
        lambda: response_mod.precompute_one_arm_wdm_dT_links(
            times_pad, orbits, EMRI_PARAMS['theta'], EMRI_PARAMS['phi'], hp, hc,
            Nf=int(cfg['Nf']), d=int(cfg['d']), L_trunc=int(cfg['L_trunc']),
            cache=one_arm_cache, variable_shift_kwargs=shift_kwargs,
            use_jax=True, q=int(cfg['q']), calc_m0=bool(cfg['calc_m0']),
        )
    )

    (_, _, _), stages['sagnac_ms'] = timed_ms(
        lambda: gen2_sagnac_wdm(
            times_pad, hp, hc, EMRI_PARAMS['theta'], EMRI_PARAMS['phi'], orbits,
            Nf=int(cfg['Nf']), shifts_cache=shifts_cache, L_trunc=int(cfg['L_trunc']),
            one_arm_cache=one_arm_cache, variable_shift_kwargs=shift_kwargs,
            use_jax=True, q=int(cfg['q']),
        )
    )

    (X1, Y1, Z1), stages['michelson_ms'] = timed_ms(
        lambda: gen2_michelson_wdm(
            times_pad, hp, hc, EMRI_PARAMS['theta'], EMRI_PARAMS['phi'], orbits,
            Nf=int(cfg['Nf']), shifts_cache=shifts_cache, L_trunc=int(cfg['L_trunc']),
            one_arm_cache=one_arm_cache, variable_shift_kwargs=shift_kwargs,
            use_jax=True, q=int(cfg['q']),
        )
    )

    stages['total_ms'] = stages['waveform_ms'] + stages['one_arm_ms'] + stages['sagnac_ms'] + stages['michelson_ms']
    return {
        'stages': stages,
        'times_pad': times_pad,
        'hplus': hp,
        'hcross': hc,
        'pad_info': pad_info,
        'wdm_obj': wdm_obj,
        'X1': X1, 'Y1': Y1, 'Z1': Z1,
    }

In [17]:
BASE_N_SAMPLES = N_SAMPLES
BASE_CFG = dict(BASELINE)

# Build waveform payload once and reuse it everywhere.
WAVEFORMS, waveform_build_ms = timed_ms(
    lambda: prebuild_waveform_payloads(BASE_CFG, [BASE_N_SAMPLES])
)
WAVEFORM_BUILD_MS = float(waveform_build_ms)

times_pad_ref, hp_ref, hc_ref, pad_info_ref = WAVEFORMS[BASE_N_SAMPLES]
BLANK_WAVEFORMS = {
    BASE_N_SAMPLES: (
        times_pad_ref,
        np.zeros_like(hp_ref),
        np.zeros_like(hc_ref),
        pad_info_ref,
    )
}

# Warmup run on same-size/same-shape blank waveform payload
warmup_run = run_pipeline(
    BASE_CFG, BASE_N_SAMPLES, waveform_payload=BLANK_WAVEFORMS[BASE_N_SAMPLES]
 )

# Measured run on waveform of interest
baseline_run = run_pipeline(
    BASE_CFG, BASE_N_SAMPLES, waveform_payload=WAVEFORMS[BASE_N_SAMPLES]
 )

# Remove legacy short-mode variables from the live kernel state if present.
for _legacy in ['SHORT_OBS_YEARS', 'SHORT_N_SAMPLES', 'SHORT_NF', 'SHORT_CFG', 'short_run']:
    globals().pop(_legacy, None)

pd.DataFrame([
    {
        'obs_years': T_OBS_YEARS,
        'n_samples': BASE_N_SAMPLES,
        'Nf': BASE_CFG['Nf'],
        'dt': BASE_CFG['dt'],
        'q': BASE_CFG['q'],
        'L_trunc': BASE_CFG['L_trunc'],
        'waveform_build_ms_once': WAVEFORM_BUILD_MS,
        'warmup_total_ms_blank': warmup_run['stages']['total_ms'],
        'runtime_total_ms': baseline_run['stages']['total_ms'],
    }
]).round(4)

,obs_years,n_samples,Nf,dt,q,L_trunc,waveform_build_ms_once,warmup_total_ms_blank,runtime_total_ms
0,0.25,788953,2000,10.0,4,25,4963.4728,37294.983,34596.8589


In [18]:
def compute_cut_wdm_mismatch(ref_td, test_wdm, wdm, cfg, pad_info, n_original):
    ref_wdm = wdm.dwt(ref_td).real
    test_use = test_wdm.real

    cut_info = compute_wdm_cut_info(
        original_length=int(n_original),
        Nf=int(cfg['Nf']),
        dt=float(cfg['dt']),
        alpha=float(cfg['alpha']),
        tukey_factor=float(cfg['tukey_factor']),
        window_pad_left=int(pad_info['window_pad_left']),
        window_pad_right=int(pad_info['window_pad_right']),
        include_taper_margin=True,
        wdm_dT=float(wdm.dT),
        wdm_Nt=int(wdm.Nt),
        drop_m0=True,
    )

    ref_cut = apply_wdm_cut(ref_wdm, cut_info)
    test_cut = apply_wdm_cut(test_use, cut_info)
    ov_cut, mm_cut = wdm_mismatch(ref_cut, test_cut)

    ref_raw = ref_wdm[:, 1:]
    test_raw = test_use[:, 1:]
    ov_raw, mm_raw = wdm_mismatch(ref_raw, test_raw)

    return {
        'overlap_cut': float(ov_cut),
        'mismatch_cut': float(mm_cut),
        'overlap_raw': float(ov_raw),
        'mismatch_raw': float(mm_raw),
    }

def warmup_response_pass(cfg, waveform_payload):
    times_pad, hp, hc, _ = waveform_payload
    shift_kwargs = build_shift_kwargs(cfg)

    # Warm only WDM kernels on blank data (no TD warmup).
    response_mod.precompute_one_arm_wdm_dT_links(
        times_pad, orbits, EMRI_PARAMS['theta'], EMRI_PARAMS['phi'], hp, hc,
        Nf=int(cfg['Nf']), d=int(cfg['d']), L_trunc=int(cfg['L_trunc']),
        cache={}, variable_shift_kwargs=shift_kwargs,
        use_jax=True, q=int(cfg['q']), calc_m0=bool(cfg['calc_m0']),
    )
    gen2_michelson_wdm(
        times_pad, hp, hc, EMRI_PARAMS['theta'], EMRI_PARAMS['phi'], orbits,
        Nf=int(cfg['Nf']), L_trunc=int(cfg['L_trunc']),
        one_arm_cache={}, shifts_cache={}, variable_shift_kwargs=shift_kwargs,
        use_jax=True, q=int(cfg['q']),
    )

def compare_one_arm_and_michelson(cfg, n_eval, waveform_payload, num_int_points=10):
    times_pad, hp, hc, pad_info = waveform_payload
    shift_kwargs = build_shift_kwargs(cfg)
    links = [(2, 1), (3, 2), (1, 3), (3, 1), (2, 3), (1, 2)]

    (wdm_payload, one_arm_wdm_ms) = timed_ms(
        lambda: response_mod.precompute_one_arm_wdm_dT_links(
            times_pad, orbits, EMRI_PARAMS['theta'], EMRI_PARAMS['phi'], hp, hc,
            Nf=int(cfg['Nf']), d=int(cfg['d']), L_trunc=int(cfg['L_trunc']),
            cache={}, variable_shift_kwargs=shift_kwargs,
            use_jax=True, q=int(cfg['q']), calc_m0=bool(cfg['calc_m0']),
        )
    )
    wdm_obj, one_arm_wdm = wdm_payload

    def _build_td_one_arm_links():
        td_map = {}
        for s, r in links:
            td_map[(s, r)] = response_mod.one_arm_response_td(
                times_pad, s, r, orbits, EMRI_PARAMS['theta'], EMRI_PARAMS['phi'], hp, hc,
                num_int_points=num_int_points,
            )
        return td_map

    (one_arm_td, one_arm_td_ms) = timed_ms(_build_td_one_arm_links)

    one_arm_rows = []
    for s, r in links:
        metrics = compute_cut_wdm_mismatch(
            one_arm_td[(s, r)], one_arm_wdm[(s, r)], wdm_obj, cfg, pad_info, n_eval
        )
        one_arm_rows.append({
            'quantity': f'dl_{s}{r}',
            'overlap_cut': metrics['overlap_cut'],
            'mismatch_cut': metrics['mismatch_cut'],
            'overlap_raw': metrics['overlap_raw'],
            'mismatch_raw': metrics['mismatch_raw'],
        })

    ((X_td, Y_td, Z_td), michelson_td_ms) = timed_ms(
        lambda: gen2_michelson_td(
            times_pad, hp, hc, EMRI_PARAMS['theta'], EMRI_PARAMS['phi'], orbits,
            num_int_points=num_int_points,
        )
    )
    ((X_w, Y_w, Z_w), michelson_wdm_ms) = timed_ms(
        lambda: gen2_michelson_wdm(
            times_pad, hp, hc, EMRI_PARAMS['theta'], EMRI_PARAMS['phi'], orbits,
            Nf=int(cfg['Nf']), L_trunc=int(cfg['L_trunc']),
            one_arm_cache={}, shifts_cache={}, variable_shift_kwargs=shift_kwargs,
            use_jax=True, q=int(cfg['q']),
        )
    )

    mx = compute_cut_wdm_mismatch(X_td, X_w, wdm_obj, cfg, pad_info, n_eval)
    my = compute_cut_wdm_mismatch(Y_td, Y_w, wdm_obj, cfg, pad_info, n_eval)
    mz = compute_cut_wdm_mismatch(Z_td, Z_w, wdm_obj, cfg, pad_info, n_eval)

    michelson_rows = [
        {'quantity': 'X', **mx},
        {'quantity': 'Y', **my},
        {'quantity': 'Z', **mz},
    ]

    timing_rows = [
        {
            'quantity': 'one_arm_6_links',
            'td_ms': float(one_arm_td_ms),
            'wdm_ms': float(one_arm_wdm_ms),
        },
        {
            'quantity': 'michelson_xyz',
            'td_ms': float(michelson_td_ms),
            'wdm_ms': float(michelson_wdm_ms),
        },
    ]
    timing = pd.DataFrame(timing_rows)
    timing['td_over_wdm_speedup'] = timing['td_ms'] / timing['wdm_ms'].clip(lower=1e-12)

    timing_stage_rows = [
        {'group': 'one_arm', 'method': 'TD', 'stage': '6_links_total', 'ms': float(one_arm_td_ms)},
        {'group': 'one_arm', 'method': 'WDM', 'stage': 'precompute_6_links', 'ms': float(one_arm_wdm_ms)},
        {'group': 'michelson', 'method': 'TD', 'stage': 'X_Y_Z_total', 'ms': float(michelson_td_ms)},
        {'group': 'michelson', 'method': 'WDM', 'stage': 'X_Y_Z_total', 'ms': float(michelson_wdm_ms)},
    ]
    if 'WAVEFORM_BUILD_MS' in globals():
        timing_stage_rows.append(
            {'group': 'waveform', 'method': 'WDM', 'stage': 'build_once', 'ms': float(WAVEFORM_BUILD_MS)}
        )
    if 'baseline_run' in globals() and isinstance(baseline_run, dict) and 'stages' in baseline_run:
        for stage_name in ['one_arm_ms', 'sagnac_ms', 'michelson_ms', 'total_ms']:
            if stage_name in baseline_run['stages']:
                timing_stage_rows.append({
                    'group': 'wdm_pipeline',
                    'method': 'WDM',
                    'stage': stage_name,
                    'ms': float(baseline_run['stages'][stage_name]),
                })
    timing_stages = pd.DataFrame(timing_stage_rows)

    return {
        'one_arm': pd.DataFrame(one_arm_rows),
        'michelson': pd.DataFrame(michelson_rows),
        'timing': timing,
        'timing_stages': timing_stages,
    }

# Warmup TD/WDM response kernels using same-shape blank waveform payload.
warmup_response_pass(BASE_CFG, BLANK_WAVEFORMS[BASE_N_SAMPLES])

# Measured comparisons on waveform of interest.
cmp = compare_one_arm_and_michelson(
    BASE_CFG, BASE_N_SAMPLES, WAVEFORMS[BASE_N_SAMPLES], num_int_points=10
)

one_arm_table = cmp['one_arm'].sort_values('mismatch_cut').reset_index(drop=True)
michelson_table = cmp['michelson'].sort_values('mismatch_cut').reset_index(drop=True)
timing_table = cmp['timing'].sort_values('quantity').reset_index(drop=True)
timing_stage_table = cmp['timing_stages'].sort_values(['group', 'method', 'stage']).reset_index(drop=True)

one_arm_table

,quantity,overlap_cut,mismatch_cut,overlap_raw,mismatch_raw
0,dl_32,0.999885,0.000115,0.999883,0.000117
1,dl_23,0.999884,0.000116,0.999882,0.000118
2,dl_31,0.999865,0.000135,0.999858,0.000142
3,dl_13,0.999865,0.000135,0.999857,0.000143
4,dl_12,0.999853,0.000147,0.999847,0.000153
5,dl_21,0.999853,0.000147,0.999847,0.000153


In [19]:
michelson_table

,quantity,overlap_cut,mismatch_cut,overlap_raw,mismatch_raw
0,X,0.999790,0.000210,0.999782,0.000218
1,Y,0.999696,0.000304,0.999687,0.000313
2,Z,0.999679,0.000321,0.999672,0.000328


In [20]:
timing_table

,quantity,td_ms,wdm_ms,td_over_wdm_speedup
0,michelson_xyz,168059.697375,22563.401667,7.448332
1,one_arm_6_links,10877.975208,4353.131084,2.498885


In [21]:
timing_stage_table

,group,method,stage,ms
0,michelson,TD,X_Y_Z_total,168059.697375
1,michelson,WDM,X_Y_Z_total,22563.401667
2,one_arm,TD,6_links_total,10877.975208
3,one_arm,WDM,precompute_6_links,4353.131084
4,waveform,WDM,build_once,4963.472833
5,wdm_pipeline,WDM,michelson_ms,17100.255000
6,wdm_pipeline,WDM,one_arm_ms,5232.112792
7,wdm_pipeline,WDM,sagnac_ms,12264.491125
8,wdm_pipeline,WDM,total_ms,34596.858917
